## Now I will do example xgb model with random parameters used. I will use it as my benchmark.

In [ ]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

In [ ]:
data = pd.read_csv("dataset.csv")
target = data["num"]
data.drop(columns="num", inplace=True)
train_x, valid_x, train_y, valid_y = train_test_split(data, target, test_size=0.20)

def objective(trial):
    param = {
        'verbosity': 0,
        'objective': 'multi:softmax',
        'num_class': 3,
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
    }

    model = xgb.XGBClassifier(**param)
    model.fit(train_x, train_y)

    preds = model.predict(valid_x)
    accuracy = accuracy_score(valid_y, preds)
    
    return accuracy

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)
print("Best score (Accuracy):", study.best_value)

best_model = xgb.XGBClassifier(**study.best_params)
best_model.fit(train_x, train_y)

In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances
plot_param_importances(study).show()